### IMPORT ALL THE REQUIRED LIBRARIES AND MODULES

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import plotly as px
from sklearn.metrics import f1_score,precision_score,recall_score,roc_auc_score,accuracy_score,confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier,RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

#### LOAD THE DATA(CSV FORMAT)

In [2]:
df=pd.read_csv('data/Travel.csv')

#### FIRST VIEW OF THE DATA 

In [3]:
df.head(6)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0
5,200005,0,32.0,Company Invited,1,8.0,Salaried,Male,3,3.0,Basic,3.0,Single,1.0,0,5,1,1.0,Executive,18068.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                4888 non-null   int64  
 1   ProdTaken                 4888 non-null   int64  
 2   Age                       4662 non-null   float64
 3   TypeofContact             4863 non-null   str    
 4   CityTier                  4888 non-null   int64  
 5   DurationOfPitch           4637 non-null   float64
 6   Occupation                4888 non-null   str    
 7   Gender                    4888 non-null   str    
 8   NumberOfPersonVisiting    4888 non-null   int64  
 9   NumberOfFollowups         4843 non-null   float64
 10  ProductPitched            4888 non-null   str    
 11  PreferredPropertyStar     4862 non-null   float64
 12  MaritalStatus             4888 non-null   str    
 13  NumberOfTrips             4748 non-null   float64
 14  Passport           

In [5]:
df.isnull().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [6]:
df['Gender']=df['Gender'].replace('Fe Male','Female')

In [7]:
df['Gender'].value_counts()

Gender
Male      2916
Female    1972
Name: count, dtype: int64

In [8]:
df['MaritalStatus'].value_counts()

MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64

In [9]:
df['MaritalStatus']=df['MaritalStatus'].replace('Single','Unmarried')

In [10]:
Feature_with_na=[features for  features in df.columns if df[features].isnull().sum()>=1]
for feature in Feature_with_na:
    print(feature,np.round(df[feature].isnull().mean()*100,5))

Age 4.62357
TypeofContact 0.51146
DurationOfPitch 5.13502
NumberOfFollowups 0.92062
PreferredPropertyStar 0.53191
NumberOfTrips 2.86416
NumberOfChildrenVisiting 1.35025
MonthlyIncome 4.76678


In [11]:
df[Feature_with_na].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,4662.0,37.622265,9.316387,18.0,31.0,36.0,44.0,61.0
DurationOfPitch,4637.0,15.490835,8.519643,5.0,9.0,13.0,20.0,127.0
NumberOfFollowups,4843.0,3.708445,1.002509,1.0,3.0,4.0,4.0,6.0
PreferredPropertyStar,4862.0,3.581037,0.798009,3.0,3.0,3.0,4.0,5.0
NumberOfTrips,4748.0,3.236521,1.849019,1.0,2.0,3.0,4.0,22.0
NumberOfChildrenVisiting,4822.0,1.187267,0.857861,0.0,1.0,1.0,2.0,3.0
MonthlyIncome,4655.0,23619.853491,5380.698361,1000.0,20346.0,22347.0,25571.0,98678.0


#### Handled Missing values

In [12]:
df['Age']=df['Age'].fillna(df['Age'].median(),inplace=True)
df['DurationOfPitch']=df['DurationOfPitch'].fillna(df['DurationOfPitch'].median(),inplace=True)
df['MonthlyIncome']=df['MonthlyIncome'].fillna(df['MonthlyIncome'].median(),inplace=True)
df['NumberOfChildrenVisiting']=df['NumberOfChildrenVisiting'].fillna(df['NumberOfChildrenVisiting'].mode()[0],inplace=True)
df['NumberOfFollowups']=df['NumberOfFollowups'].fillna(df['NumberOfFollowups'].mode()[0],inplace=True)
df['PreferredPropertyStar']=df['PreferredPropertyStar'].fillna(df['PreferredPropertyStar'].mode()[0],inplace=True)
df['NumberOfTrips']=df['NumberOfTrips'].fillna(df['NumberOfTrips'].median(),inplace=True)
df['TypeofContact']=df['TypeofContact'].fillna(df['TypeofContact'].mode()[0],inplace=True)


In [13]:
df.isnull().sum()

CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

In [14]:
df.drop(columns=['CustomerID'],inplace=True)

In [15]:
df['TotalNumberPeopleVisiting']=df['NumberOfPersonVisiting']+df['NumberOfChildrenVisiting']

In [16]:
df.drop(columns=['NumberOfPersonVisiting','NumberOfChildrenVisiting'],inplace=True)

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ProdTaken                  4888 non-null   int64  
 1   Age                        4888 non-null   float64
 2   TypeofContact              4888 non-null   str    
 3   CityTier                   4888 non-null   int64  
 4   DurationOfPitch            4888 non-null   float64
 5   Occupation                 4888 non-null   str    
 6   Gender                     4888 non-null   str    
 7   NumberOfFollowups          4888 non-null   float64
 8   ProductPitched             4888 non-null   str    
 9   PreferredPropertyStar      4888 non-null   float64
 10  MaritalStatus              4888 non-null   str    
 11  NumberOfTrips              4888 non-null   float64
 12  Passport                   4888 non-null   int64  
 13  PitchSatisfactionScore     4888 non-null   int64  
 14  Own

In [18]:
num_feature=[features for features in df.columns if df[features].dtypes!='str']
print('Number of numerical feature:',len(num_feature))
cat_features=[features for features in df.columns  if df[features].dtypes=='str']
print("Number of categoraical features:",len(cat_features))
desc_features=[features for features in num_feature if len(df[features].unique())<25]
print('number of Descrite features:',len(desc_features))
Continous_features=[features for features in num_feature if features not in desc_features]
print('Number of continous features:',len(Continous_features))

Number of numerical feature: 12
Number of categoraical features: 6
number of Descrite features: 9
Number of continous features: 3


In [19]:
x=df.drop(['ProdTaken'],axis=True)
y=df['ProdTaken']

In [20]:
y

0       1
1       0
2       1
3       0
4       0
       ..
4883    1
4884    1
4885    1
4886    1
4887    1
Name: ProdTaken, Length: 4888, dtype: int64

In [21]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=10)


In [22]:
cat_features=x.select_dtypes(include='str').columns
num_feature=x.select_dtypes(exclude='str').columns

In [23]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer


In [24]:
num_transform=StandardScaler()
cat_transform=OneHotEncoder(drop='first')

In [25]:
processor=ColumnTransformer(
    [
        ('OneHotEncoder',cat_transform,cat_features),
        ('StandardScaler',num_transform,num_feature)
    ]
)

In [26]:
x_train=processor.fit_transform(x_train)
x_test=processor.transform(x_test)

In [30]:
models={
    'RandomForest':RandomForestClassifier(),
    'DecisionTree':DecisionTreeClassifier(),
    'Logistic':LogisticRegression(),
    'KNeighbors':KNeighborsClassifier(),
    'GradientBoosting':GradientBoostingClassifier()
}
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(x_train,y_train)
    y_train_pred=model.predict(x_train)
    y_test_pred=model.predict(x_test)
    train_accuracy=accuracy_score(y_train,y_train_pred)
    train_f1=f1_score(y_train,y_train_pred,average='weighted')
    train_precission=precision_score(y_train,y_train_pred)
    train_recall=recall_score(y_train,y_train_pred)
    roc_auc_score_train=roc_auc_score(y_train,y_train_pred)
    test_accuracy=accuracy_score(y_test,y_test_pred)
    test_f1=f1_score(y_test,y_test_pred,average='weighted')
    test_precission=precision_score(y_test,y_test_pred)
    test_recall=recall_score(y_test,y_test_pred)
    roc_auc_score_test=roc_auc_score(y_test,y_test_pred)
    print(list(models.keys())[i])
    print('Model perforance for training set')
    print("- Accuracy:{:.4f}".format(train_accuracy))
    print("- precision_score:{:.4f}".format(train_precission))
    print("- recall_score:{:.4f}".format(train_recall))
    print("- f1_score: {:.4f}".format(train_f1))
    print("- roc_auc_score:{:.4f}".format(roc_auc_score_train))

    print("----------------------------------------------------")

    print("Model performance for test")
    print("- Accuracy:{:.4f}".format(test_accuracy))
    print("- precision_score:{:.4f}".format(test_precission))
    print("- recall_score:{:.4f}".format(test_recall))
    print("- roc_auc_score:{:.4f}".format(roc_auc_score_test))
    print("- f1_score:{:.4f}".format(test_f1))

    print('='*35)
    print('\n')

RandomForest
Model perforance for training set
- Accuracy:1.0000
- precision_score:1.0000
- recall_score:1.0000
- f1_score: 1.0000
- roc_auc_score:1.0000
----------------------------------------------------
Model performance for test
- Accuracy:0.9284
- precision_score:0.9333
- recall_score:0.6087
- roc_auc_score:0.8001
- f1_score:0.9221


DecisionTree
Model perforance for training set
- Accuracy:1.0000
- precision_score:1.0000
- recall_score:1.0000
- f1_score: 1.0000
- roc_auc_score:1.0000
----------------------------------------------------
Model performance for test
- Accuracy:0.9090
- precision_score:0.6915
- recall_score:0.8075
- roc_auc_score:0.8682
- f1_score:0.9118


Logistic
Model perforance for training set
- Accuracy:0.8460
- precision_score:0.7116
- recall_score:0.3478
- f1_score: 0.8241
- roc_auc_score:0.6569
----------------------------------------------------
Model performance for test
- Accuracy:0.8517
- precision_score:0.5930
- recall_score:0.3168
- roc_auc_score:0.637

In [31]:
rand={
    "max_depth": [5, 8, 15, None, 10],
    "max_features": [5, 7, "auto", 8],
    "min_samples_split": [2, 8, 15, 20],
    "n_estimators": [100, 200, 500, 1000]
}

In [32]:
rand_cv=[
    ('RF',RandomForestClassifier(),rand)
]

In [33]:
from sklearn.model_selection import RandomizedSearchCV
model_params={}
for name,model,params in rand_cv:
    random=RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=100,
        cv=3,
        verbose=2,
        n_jobs=-1
    )
    random.fit(x_train, y_train)
    model_params[name] = random.best_params_

for model_name in model_params:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_params[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits
---------------- Best Params for RF -------------------
{'n_estimators': 200, 'min_samples_split': 2, 'max_features': 8, 'max_depth': None}


In [34]:
models={
'RandomForest':RandomForestClassifier(n_estimators= 200, min_samples_split= 2, max_features= 8, max_depth= None)
}
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(x_train,y_train)
    y_train_pred=model.predict(x_train)
    y_test_pred=model.predict(x_test)
    train_accuracy=accuracy_score(y_train,y_train_pred)
    train_f1=f1_score(y_train,y_train_pred,average='weighted')
    train_precission=precision_score(y_train,y_train_pred)
    train_recall=recall_score(y_train,y_train_pred)
    roc_auc_score_train=roc_auc_score(y_train,y_train_pred)
    test_accuracy=accuracy_score(y_test,y_test_pred)
    test_f1=f1_score(y_test,y_test_pred,average='weighted')
    test_precission=precision_score(y_test,y_test_pred)
    test_recall=recall_score(y_test,y_test_pred)
    roc_auc_score_test=roc_auc_score(y_test,y_test_pred)
    print(list(models.keys())[i])
    print('Model perforance for training set')
    print("- Accuracy:{:.4f}".format(train_accuracy))
    print("- precision_score:{:.4f}".format(train_precission))
    print("- recall_score:{:.4f}".format(train_recall))
    print("- f1_score: {:.4f}".format(train_f1))
    print("- roc_auc_score:{:.4f}".format(roc_auc_score_train))

    print("----------------------------------------------------")

    print("Model performance for test")
    print("- Accuracy:{:.4f}".format(test_accuracy))
    print("- precision_score:{:.4f}".format(test_precission))
    print("- recall_score:{:.4f}".format(test_recall))
    print("- roc_auc_score:{:.4f}".format(roc_auc_score_test))
    print("- f1_score:{:.4f}".format(test_f1))

    print('='*35)
    print('\n')

RandomForest
Model perforance for training set
- Accuracy:1.0000
- precision_score:1.0000
- recall_score:1.0000
- f1_score: 1.0000
- roc_auc_score:1.0000
----------------------------------------------------
Model performance for test
- Accuracy:0.9417
- precision_score:0.9262
- recall_score:0.7019
- roc_auc_score:0.8454
- f1_score:0.9384


